# Notebook D v4.3: gold_person_signal_scores + gold_person_scores

**Run order:** After Notebook A v4.2 and Notebook B v4.2.

## v4.3 changes vs v4.2

DNA signal support added to `gold_person_signal_scores`:

- Three new applicability flag handlers added to the `CASE` block:
  - `requires_dna_match_tag` — FALSE if person does not have DNA Match tag (`@T10@`)
  - `requires_dna_ancestor_tag` — FALSE if person does not have Common DNA Ancestor tag (`@T5@`)
  - `requires_generation_lte` — FALSE if generation depth exceeds N
- Four new signal codes added to `UNPIVOT IN (...)` list:
  - `SIGNAL_DNA_CITATION_MISSING`
  - `SIGNAL_DNA_PATH_INCOMPLETE`
  - `SIGNAL_NO_DNA_CORROBORATION`
  - `SIGNAL_COMMON_ANCESTOR_UNVERIFIED`

No changes to `gold_person_scores`, `gold_branch_scores`, or score formulas.

## Architecture

### gold_person_signal_scores
Long-format view — one row per person x signal.
Joins `gold_research_person_signals` (detection booleans + context columns)
against `ref_signal_weights` (applicability flags + base scores).
Evaluates applicability per person per signal.
Exposes: `is_applicable`, `is_fired`, `base_score`, `dimension`.

**Adding a new signal in future requires:**
1. Add boolean column to Notebook B (signals view)
2. Insert one row into `ref_signal_weights` with applicability flags
3. Add to the UNPIVOT in cell 1 of this notebook

### gold_person_scores (v4.2)
Aggregates `gold_person_signal_scores` into absolute 0-100 scores.
Also computes proximity+depth weighted scores for use in ranking/prioritisation.

**Absolute scores** (completeness, evidence, overall, story_potential):
- Unchanged from v4.1 -- no person's score is affected by another's
- Used for story_ready threshold and Scores tab display

**Weighted scores** (weighted_completeness, weighted_evidence,
weighted_overall, weighted_story_potential):
- Absolute score x proximity_multiplier x depth_multiplier
- Used for ordering in This Week cards, branch priority, chat tools
- Direct ancestors with research gaps surface above distant relatives
  with the same raw score

### Multiplier values
```
proximity (steps from direct ancestral line):
  0 -> 1.00   (direct ancestor)
  1 -> 1.25   (close collateral -- sibling/child of ancestor)
  2 -> 1.50   (extended collateral)
  3+ -> 2.00  (distant)

depth (generations from researcher to common ancestor):
  <= 4  -> 1.00
  5-6   -> 1.10
  7-8   -> 1.20
  9+    -> 1.40
```

### gold_branch_scores
Unchanged in structure -- aggregates gold_person_scores.

## Score formula
```
completeness_score = (comp_applicable - comp_fired) / comp_applicable * 100
evidence_score     = (evid_applicable - evid_fired) / evid_applicable * 100
story_score        = PERCENT_RANK() OVER (ORDER BY narr_fired) * 100
overall_score      = completeness * 0.40 + evidence * 0.60
```

## Notes on applicability
- SIGNAL_MISSING_BURIAL: `requires_death_recorded` flag used with implicit
  `death_year >= 1837` check (handled in the CASE WHEN in the view)
- SIGNAL_SINGLE_SOURCE_DEPENDENCE: `requires_lifespan_gte` repurposed as
  event_count proxy (>= 3 events approx >= 3 facts). Handled explicitly.
- LOW and VERY_LOW evidence density: both always in denominator;
  only one fires at a time (mutually exclusive at the detection level)
- DNA signals: applicability is tag-based. DNA signals only count toward
  a person's applicable maximum if they have the relevant DNA tag.
  This prevents DNA Match signals unfairly penalising non-match people.


In [0]:

%sql
-- ============================================================
-- CELL 1: gold_person_signal_scores (v4.3)
--
-- v4.3 changes vs v4.2:
--   NEW applicability flag handlers (before ELSE TRUE):
--     requires_dna_match_tag    -- only applicable for DNA Match tagged people
--     requires_dna_ancestor_tag -- only applicable for Common DNA Ancestor tagged people
--     requires_generation_lte   -- only applicable if generation_depth <= N
--   NEW signals added to UNPIVOT IN (...):
--     SIGNAL_DNA_CITATION_MISSING, SIGNAL_DNA_PATH_INCOMPLETE,
--     SIGNAL_NO_DNA_CORROBORATION, SIGNAL_COMMON_ANCESTOR_UNVERIFIED
--
-- One row per person × signal from ref_signal_weights.
-- Evaluates applicability flags against person context columns
-- exposed by gold_research_person_signals (v4).
--
-- is_applicable: TRUE if this signal's conditions are met for this person
-- is_fired:      TRUE if the signal boolean is TRUE in the signals view
--
-- To add a new signal:
--   1. Add detection boolean to Notebook B
--   2. Add row to ref_signal_weights with applicability flags
--   3. Add one line to the UNPIVOT IN (...) list below
--   4. Done — scoring and calibration views update automatically
-- ============================================================

CREATE OR REPLACE VIEW genealogy.gold_person_signal_scores AS

WITH

-- Unpivot the wide boolean signals view into long format.
-- One row per person per signal that appears in ref_signal_weights.
-- INCLUDE NULLS: retain FALSE values (default EXCLUDE NULLS would drop them).
-- To add a new signal: add one column name to the IN (...) list.
unpivoted AS (
  SELECT person_gedcom_id, signal_code, is_fired
  FROM genealogy.gold_research_person_signals
  UNPIVOT INCLUDE NULLS (
    is_fired FOR signal_code IN (
      SIGNAL_NO_BIRTH_RECORDED,
      SIGNAL_NO_DEATH_RECORDED,
      SIGNAL_NO_MARRIAGES,
      SIGNAL_NO_CHILDREN,
      SIGNAL_MISSING_PARENT,
      SIGNAL_MISSING_CENSUS_COVERAGE,
      SIGNAL_UNCOVERED_SOURCES,
      SIGNAL_DOCS_NOT_TRANSCRIBED,
      SIGNAL_LATE_LIFE_GAP,
      SIGNAL_EARLY_LIFE_ONLY,
      SIGNAL_CHILD_GAPS,
      SIGNAL_UNCONFIRMED_MILITARY,
      SIGNAL_MISSING_OCCUPATION,
      SIGNAL_MISSING_BURIAL,
      SIGNAL_NO_RESIDENCE,
      SIGNAL_NO_DOCUMENTS_AT_ALL,
      SIGNAL_FACT_CONFLICT,
      SIGNAL_TRANSCRIPT_ONLY_FACTS,
      SIGNAL_VERY_LOW_EVIDENCE_DENSITY,
      SIGNAL_LOW_EVIDENCE_DENSITY,
      SIGNAL_SINGLE_SOURCE_DEPENDENCE,
      SIGNAL_UNSOURCED_FAMILY_EVENTS,
      SIGNAL_INCOMPLETE_NAME,
      SIGNAL_IMPRECISE_DATES,
      SIGNAL_IMPRECISE_PLACES,
      SIGNAL_MIGRANT,
      SIGNAL_CONFIRMED_MILITARY,
      SIGNAL_MULTIPLE_SPOUSES,
      SIGNAL_YOUNG_DEATH,
      SIGNAL_LARGE_FAMILY,
      SIGNAL_NEWSPAPER_MENTION,
      SIGNAL_WILL_OR_PROBATE,
      SIGNAL_STORY_WRITTEN,
      SIGNAL_TRANSCRIPT_RICH,
      SIGNAL_VARIED_OCCUPATIONS,
      SIGNAL_POSSIBLE_RESIDENCE,
      SIGNAL_POSSIBLE_MARRIAGE,
      SIGNAL_POSSIBLE_CHILDREN,
      -- DNA signals (NEW in v4.3)
      SIGNAL_DNA_CITATION_MISSING,
      SIGNAL_DNA_PATH_INCOMPLETE,
      SIGNAL_NO_DNA_CORROBORATION,
      SIGNAL_COMMON_ANCESTOR_UNVERIFIED
    )
  )
)

SELECT
  u.person_gedcom_id,
  u.signal_code,
  w.dimension,
  w.category,
  w.intent,
  w.base_score,
  w.reason_label,
  u.is_fired,

  -- ── Applicability evaluation ──────────────────────────────────────────
  -- Evaluates ref_signal_weights flags against person context columns
  -- from gold_research_person_signals (v4).
  -- A signal is applicable if applies_always=TRUE OR all non-NULL flags pass.
  CASE
    WHEN w.applies_always = TRUE THEN TRUE

    -- birth year range
    WHEN w.min_birth_year IS NOT NULL
      AND (s.birth_year IS NULL OR s.birth_year < w.min_birth_year) THEN FALSE
    WHEN w.max_birth_year IS NOT NULL
      AND (s.birth_year IS NULL OR s.birth_year > w.max_birth_year) THEN FALSE

    -- sex restriction
    WHEN w.sex_restriction IS NOT NULL
      AND s.sex != w.sex_restriction THEN FALSE

    -- must be expected to have died (birth_year <= 1930 AND expected_end_year < now)
    WHEN w.requires_expected_to_die = TRUE
      AND NOT (
        (s.birth_year IS NULL OR s.birth_year <= 1930)
        AND s.expected_end_year < year(current_date)
      ) THEN FALSE

    -- lifespan threshold (used by SIGNAL_NO_RESIDENCE: effective_span_years >= 20)
    WHEN w.requires_lifespan_gte IS NOT NULL
      AND COALESCE(s.effective_span_years, 0) < w.requires_lifespan_gte THEN FALSE

    -- survived past 40 (used by SIGNAL_LATE_LIFE_GAP)
    WHEN w.requires_survived_past_40 = TRUE
      AND NOT (
        s.birth_year IS NOT NULL
        AND s.expected_end_year >= s.birth_year + 40
      ) THEN FALSE

    -- working age (used by SIGNAL_MISSING_OCCUPATION: male, end_year >= birth + 18)
    WHEN w.requires_working_age = TRUE
      AND NOT (
        s.birth_year IS NOT NULL
        AND s.expected_end_year >= s.birth_year + 18
      ) THEN FALSE

    -- at least one expected census year (used by SIGNAL_MISSING_CENSUS_COVERAGE)
    WHEN w.requires_census_years = TRUE
      AND COALESCE(s.total_expected_censuses, 0) = 0 THEN FALSE

    -- post-1837 birth, death, or marriage (used by SIGNAL_IMPRECISE_DATES)
    WHEN w.requires_post_1837_event = TRUE
      AND NOT (
        COALESCE(s.birth_year, 0) >= 1837
        OR COALESCE(s.death_year, 0) >= 1837
        OR COALESCE(s.earliest_marriage_year, 0) >= 1837
      ) THEN FALSE

    -- has at least one matched document
    -- (used by SIGNAL_DOCS_NOT_TRANSCRIBED — can't have untranscribed docs if no docs)
    WHEN w.requires_has_document = TRUE
      AND s.SIGNAL_NO_DOCUMENTS_AT_ALL = TRUE THEN FALSE

    -- has at least one transcript
    -- (used by SIGNAL_FACT_CONFLICT, SIGNAL_TRANSCRIPT_ONLY_FACTS)
    WHEN w.requires_transcript = TRUE
      AND s.has_any_transcript = FALSE THEN FALSE

    -- has at least one citation in gold_source_coverage
    -- (used by SIGNAL_UNCOVERED_SOURCES)
    WHEN w.requires_citations = TRUE
      AND s.has_citations = FALSE THEN FALSE

    -- has family events — marriages or child births
    -- (used by SIGNAL_EARLY_LIFE_ONLY, SIGNAL_POSSIBLE_RESIDENCE)
    WHEN w.requires_family_events = TRUE
      AND COALESCE(s.num_marriages, 0) = 0
      AND COALESCE(s.num_child_births, 0) = 0 THEN FALSE

    -- proximity restriction (used by SIGNAL_NO_CHILDREN: proximity <= 1)
    WHEN w.requires_proximity_lte IS NOT NULL
      AND COALESCE(s.proximity, 99) > w.requires_proximity_lte THEN FALSE

    -- death recorded AND post-1837 (used by SIGNAL_MISSING_BURIAL)
    WHEN w.requires_death_recorded = TRUE
      AND (s.death_year IS NULL OR s.death_year < 1837) THEN FALSE

    -- total_facts threshold (used by SIGNAL_SINGLE_SOURCE_DEPENDENCE: total_facts >= 3)
    WHEN w.requires_total_facts_gte IS NOT NULL
      AND COALESCE(s.total_facts, 0) < w.requires_total_facts_gte THEN FALSE

    -- suppress for young deaths (effective_span_years 16-40)
    -- (used by SIGNAL_NO_MARRIAGES, SIGNAL_NO_CHILDREN, SIGNAL_MULTIPLE_SPOUSES)
    WHEN w.not_young_death = TRUE
      AND s.death_year IS NOT NULL
      AND s.effective_span_years BETWEEN 16 AND 40 THEN FALSE

    -- ── DNA applicability flags (NEW in v4.3) ──────────────────────────────

    -- requires_dna_match_tag: only applicable for DNA Match tagged people.
    -- Used by SIGNAL_DNA_CITATION_MISSING and SIGNAL_DNA_PATH_INCOMPLETE.
    -- Ensures these signals only enter the denominator for tagged people.
    WHEN w.requires_dna_match_tag = TRUE
      AND COALESCE(s.has_dna_match_tag, FALSE) = FALSE THEN FALSE

    -- requires_dna_ancestor_tag: only applicable for Common DNA Ancestor tagged people.
    -- Used by SIGNAL_COMMON_ANCESTOR_UNVERIFIED.
    WHEN w.requires_dna_ancestor_tag = TRUE
      AND COALESCE(s.has_dna_ancestor_tag, FALSE) = FALSE THEN FALSE

    -- requires_generation_lte: only applicable if generation depth <= N.
    -- Used by SIGNAL_NO_DNA_CORROBORATION (gen <= 6).
    -- NULL generation_depth means not in gold_generation_depth -- not applicable.
    WHEN w.requires_generation_lte IS NOT NULL
      AND COALESCE(s.generation_depth, 999) > w.requires_generation_lte THEN FALSE

    ELSE TRUE
  END AS is_applicable

FROM unpivoted u
JOIN genealogy.ref_signal_weights w
  ON w.signal_code = u.signal_code
JOIN genealogy.gold_research_person_signals s
  ON s.person_gedcom_id = u.person_gedcom_id;


In [0]:
%sql
-- ============================================================
-- CELL 2: gold_person_scores (v4.2)
--
-- v4.2 change: reinstates proximity+depth weighted scores.
-- Absolute scores (completeness, evidence, overall, story_potential)
-- are UNCHANGED from v4.1 — still used for story_ready threshold
-- and display in the Scores tab.
--
-- Four new weighted columns added for ranking/prioritisation:
--   weighted_completeness_score
--   weighted_evidence_score
--   weighted_overall_score
--   weighted_story_potential_score
--
-- These should be used in ORDER BY clauses in:
--   get_week_actions (This Week cards)
--   get_branch_priority (chat tool)
--   Any other ranked query surfacing people to work on
--
-- story_ready threshold intentionally uses absolute scores —
-- it is a quality gate, not a priority ranking.
-- ============================================================

CREATE OR REPLACE VIEW genealogy.gold_person_scores AS

WITH scores AS (
  SELECT
    person_gedcom_id,

    -- Completeness: applicable max and fired sum
    SUM(CASE WHEN dimension = 'completeness' AND is_applicable AND base_score > 0
             THEN base_score ELSE 0 END) AS comp_applicable,
    SUM(CASE WHEN dimension = 'completeness' AND is_applicable AND is_fired AND base_score > 0
             THEN base_score ELSE 0 END) AS comp_fired,

    -- Evidence: applicable max and fired sum
    SUM(CASE WHEN dimension = 'evidence' AND is_applicable AND base_score > 0
             THEN base_score ELSE 0 END) AS evid_applicable,
    SUM(CASE WHEN dimension = 'evidence' AND is_applicable AND is_fired AND base_score > 0
             THEN base_score ELSE 0 END) AS evid_fired,

    -- Narrative: exclude SIGNAL_STORY_WRITTEN from denominator entirely
    SUM(CASE WHEN dimension = 'narrative' AND is_applicable AND base_score > 0
               AND signal_code != 'SIGNAL_STORY_WRITTEN'
             THEN base_score ELSE 0 END) AS narr_applicable,
    SUM(CASE WHEN dimension = 'narrative' AND is_applicable AND is_fired AND base_score > 0
               AND signal_code != 'SIGNAL_STORY_WRITTEN'
             THEN base_score ELSE 0 END) AS narr_fired,

    -- Story written flag (sourced from signal firing, used as fallback)
    MAX(CASE WHEN signal_code = 'SIGNAL_STORY_WRITTEN' AND is_fired THEN TRUE ELSE FALSE END)
      AS story_written_flag

  FROM genealogy.gold_person_signal_scores
  GROUP BY person_gedcom_id
),

-- proximity: minimum ancestral_proximity across all paths to researcher
-- (a person can appear via multiple routes; take the closest)
prox AS (
  SELECT person_id, MIN(ancestral_proximity) AS proximity
  FROM genealogy.gold_ancestral_proximity
  GROUP BY person_id
),

story AS (
  SELECT person_gedcom_id, story_written, story_title, story_doc_id
  FROM genealogy.silver_person_story_status
),

-- depth: from gold_research_person_signals (same source as proximity).
-- Joined separately here so prox CTE stays clean and reusable.
-- depth = generations from researcher to the common ancestor.
depth_cte AS (
  SELECT person_gedcom_id, depth
  FROM genealogy.gold_research_person_signals
),

-- Pre-compute multiplier components for clarity and reuse.
-- proximity uses prox.proximity (same value as signals.proximity — confirmed).
-- depth uses the signals view directly.
multipliers AS (
  SELECT
    pr.person_id AS person_gedcom_id,

    -- Proximity multiplier: how close to the direct ancestral line?
    --   0 = direct ancestor          → boost strongly
    --   1 = close collateral          → moderate boost
    --   2 = extended collateral       → neutral
    --   3+ = distant relative         → slight penalty
    CASE
      WHEN pr.proximity = 0 THEN 1.00
      WHEN pr.proximity = 1 THEN 1.25
      WHEN pr.proximity = 2 THEN 1.50
      ELSE                       2.00
    END AS proximity_multiplier,

    -- Depth multiplier: how many generations back is the common ancestor?
    --   <= 4 generations = grandparents/great-grandparents → strong boost
    --   5–6 generations  = 3×/4× great-grandparents       → moderate boost
    --   7–8 generations  = deeper but still in scope       → small boost
    --   9+  generations  = very deep tree                  → neutral
    CASE
      WHEN d.depth <= 4              THEN 1.00
      WHEN d.depth BETWEEN 5 AND 6   THEN 1.10
      WHEN d.depth BETWEEN 7 AND 8   THEN 1.20
      ELSE                                1.40
    END AS depth_multiplier

  FROM prox pr
  LEFT JOIN depth_cte d ON d.person_gedcom_id = pr.person_id
),

-- Absolute scores and percentile input — all joins happen here.
-- Weighted score columns computed inline using multipliers.
base AS (
  SELECT
    p.person_gedcom_id,
    p.given_name,
    p.first_name,
    p.surname,
    p.display_name,
    p.birth_year,
    p.death_year,
    p.sex,
    sc.comp_applicable,
    sc.comp_fired,
    sc.evid_applicable,
    sc.evid_fired,
    sc.narr_applicable,
    sc.narr_fired,
    sc.story_written_flag,
    COALESCE(st.story_written, FALSE)  AS story_written,
    st.story_title,
    st.story_doc_id,
    pr.proximity                       AS ancestral_proximity,
    b.branch,

    -- Multiplier components (exposed for diagnostics)
    COALESCE(m.proximity_multiplier, 0.80) AS proximity_multiplier,
    COALESCE(m.depth_multiplier,     1.00) AS depth_multiplier,

    -- ── Absolute scores (unchanged from v4.1) ────────────────────────────
    -- These are the source-of-truth scores displayed in the UI and used
    -- for the story_ready threshold. Fixing one person never affects another.
    ROUND((sc.comp_applicable - sc.comp_fired) / NULLIF(sc.comp_applicable, 0) * 100, 0)
      AS completeness_score,
    ROUND((sc.evid_applicable - sc.evid_fired) / NULLIF(sc.evid_applicable, 0) * 100, 0)
      AS evidence_score,
    ROUND(
      ROUND((sc.comp_applicable - sc.comp_fired) / NULLIF(sc.comp_applicable, 0) * 100, 0) * 0.40
      + ROUND((sc.evid_applicable - sc.evid_fired) / NULLIF(sc.evid_applicable, 0) * 100, 0) * 0.60
    , 0) AS overall_score,

    -- ── Weighted scores (new in v4.2) ─────────────────────────────────────
    -- Used for ORDER BY in priority queries — not displayed as scores in UI.
    -- Applied per dimension so completeness and evidence can be ranked
    -- independently (e.g. This Week housekeeping vs tree_building modes).
    -- NULLIF guards: if comp/evid applicable = 0, weighted score stays NULL.
    ROUND(
      ROUND((sc.comp_applicable - sc.comp_fired) / NULLIF(sc.comp_applicable, 0) * 100, 0)
      * COALESCE(m.proximity_multiplier, 2.00)
      * COALESCE(m.depth_multiplier,     1.40)
    , 1) AS weighted_completeness_score,
    ROUND(
      ROUND((sc.evid_applicable - sc.evid_fired) / NULLIF(sc.evid_applicable, 0) * 100, 0)
      * COALESCE(m.proximity_multiplier, 2.00)
      * COALESCE(m.depth_multiplier,     1.40)
    , 1) AS weighted_evidence_score,
    ROUND(
      (
        ROUND((sc.comp_applicable - sc.comp_fired) / NULLIF(sc.comp_applicable, 0) * 100, 0) * 0.40
        + ROUND((sc.evid_applicable - sc.evid_fired) / NULLIF(sc.evid_applicable, 0) * 100, 0) * 0.60
      )
      * COALESCE(m.proximity_multiplier, 2.00)
      * COALESCE(m.depth_multiplier,     1.40)
    , 1) AS weighted_overall_score
    -- Note: weighted_story_potential_score computed in final SELECT
    -- after PERCENT_RANK() is applied in the ranked CTE.

  FROM genealogy.gold_person_life p
  JOIN scores sc                            ON sc.person_gedcom_id = p.person_gedcom_id
  LEFT JOIN genealogy.gold_person_branch b  ON b.person_gedcom_id  = p.person_gedcom_id
  LEFT JOIN prox pr                         ON pr.person_id         = p.person_gedcom_id
  LEFT JOIN story st                        ON st.person_gedcom_id  = p.person_gedcom_id
  LEFT JOIN multipliers m                   ON m.person_gedcom_id   = p.person_gedcom_id
),

-- Percentile rank over narr_fired — relative narrative potential.
-- People with story already written are included in the window function
-- (so percentiles are stable) but their score is zeroed in the final SELECT.
ranked AS (
  SELECT
    person_gedcom_id,
    ROUND(PERCENT_RANK() OVER (ORDER BY narr_fired) * 100, 0) AS story_potential_score
  FROM base
)

SELECT
  b.person_gedcom_id,
  b.given_name,
  b.first_name,
  b.surname,
  b.display_name,
  b.birth_year,
  b.death_year,
  b.branch,
  b.sex,

  -- Raw score components for diagnostics
  b.comp_applicable,
  b.comp_fired,
  b.evid_applicable,
  b.evid_fired,
  b.narr_applicable,
  b.narr_fired,

  -- ── Absolute scores (display + story_ready gate) ──────────────────────
  b.completeness_score,
  b.evidence_score,
  b.overall_score,

  -- Story potential: percentile rank; 0 if story already written
  CASE WHEN b.story_written_flag OR b.story_written
       THEN 0
       ELSE r.story_potential_score
  END AS story_potential_score,

  -- story_ready: uses absolute scores — quality gate, not ranking
  CASE
    WHEN b.story_written_flag OR b.story_written THEN FALSE
    WHEN r.story_potential_score >= 90 AND b.overall_score >= 70 THEN TRUE
    ELSE FALSE
  END AS story_ready,

  -- ── Weighted scores (ranking / priority ordering only) ─────────────────
  -- Use these in ORDER BY for This Week cards, branch priority, etc.
  -- Do NOT display these as scores in the UI — they exceed 100.
  b.weighted_completeness_score,
  b.weighted_evidence_score,
  b.weighted_overall_score,

  -- Weighted story potential: same zero-out logic as absolute version
  CASE WHEN b.story_written_flag OR b.story_written
       THEN 0
       ELSE ROUND(
         r.story_potential_score
         * (2 - b.proximity_multiplier) --effectively zeroes out if proximity > 2
         * (2 - b.depth_multiplier)
       , 1)
  END AS weighted_story_potential_score,

  -- Multipliers exposed for diagnostics / spot-checking
  b.proximity_multiplier,
  b.depth_multiplier,

  -- Story flags
  b.story_written,
  b.story_title,
  b.story_doc_id,

  -- Proximity
  b.ancestral_proximity,
  CASE
    WHEN b.ancestral_proximity = 0             THEN 'Direct Ancestor'
    WHEN b.ancestral_proximity = 1             THEN 'Close'
    WHEN b.ancestral_proximity BETWEEN 2 AND 3 THEN 'Collateral'
    ELSE                                            'Distant'
  END AS proximity_label

FROM base b
JOIN ranked r ON r.person_gedcom_id = b.person_gedcom_id;


In [0]:

%sql
CREATE OR REPLACE VIEW genealogy.gold_branch_scores AS
SELECT
  branch,
  COUNT(*)                                                        AS total_individuals,
  ROUND(AVG(completeness_score))                                  AS avg_completeness,
  ROUND(AVG(evidence_score))                                      AS avg_evidence,
  ROUND(AVG(story_potential_score))                               AS avg_story_potential,
  ROUND(AVG(overall_score))                                       AS avg_overall,
  SUM(CASE WHEN story_written            THEN 1 ELSE 0 END)       AS stories_written,
  SUM(CASE WHEN story_ready
            AND NOT story_written        THEN 1 ELSE 0 END)       AS stories_ready,
  COUNT(CASE WHEN ancestral_proximity = 0 THEN 1 END)             AS direct_ancestors,
  ROUND(AVG(CASE WHEN ancestral_proximity = 0
                 THEN overall_score END))                         AS ancestor_avg_overall,
  MIN(completeness_score)                                         AS min_completeness,
  MIN(evidence_score)                                             AS min_evidence
FROM genealogy.gold_person_scores
WHERE branch IS NOT NULL
GROUP BY branch
ORDER BY avg_overall DESC;


In [0]:

%sql
-- Should all return 0 violations
SELECT 'comp_fired > comp_applicable'  AS check_name, COUNT(*) AS violations
FROM genealogy.gold_person_scores WHERE comp_fired > comp_applicable
UNION ALL
SELECT 'evid_fired > evid_applicable',  COUNT(*) FROM genealogy.gold_person_scores WHERE evid_fired > evid_applicable
UNION ALL
SELECT 'narr_fired > narr_applicable',  COUNT(*) FROM genealogy.gold_person_scores WHERE narr_fired > narr_applicable
UNION ALL
SELECT 'NULL completeness_score',       COUNT(*) FROM genealogy.gold_person_scores WHERE completeness_score IS NULL
UNION ALL
SELECT 'NULL evidence_score',           COUNT(*) FROM genealogy.gold_person_scores WHERE evidence_score IS NULL
UNION ALL
SELECT 'NULL story_potential_score',    COUNT(*) FROM genealogy.gold_person_scores WHERE story_potential_score IS NULL
UNION ALL
SELECT 'comp_applicable = 0',           COUNT(*) FROM genealogy.gold_person_scores WHERE comp_applicable = 0
UNION ALL
SELECT 'evid_applicable = 0',           COUNT(*) FROM genealogy.gold_person_scores WHERE evid_applicable = 0;


In [0]:

%sql
-- Score distributions
SELECT
  'completeness'    AS dimension,
  ROUND(MIN(completeness_score))                        AS min,
  ROUND(PERCENTILE(completeness_score, 0.25))           AS p25,
  ROUND(PERCENTILE(completeness_score, 0.50))           AS median,
  ROUND(PERCENTILE(completeness_score, 0.75))           AS p75,
  ROUND(MAX(completeness_score))                        AS max,
  ROUND(AVG(completeness_score), 1)                     AS mean,
  SUM(CASE WHEN completeness_score = 100 THEN 1 END)    AS at_100,
  SUM(CASE WHEN completeness_score = 0   THEN 1 END)    AS at_0
FROM genealogy.gold_person_scores
UNION ALL
SELECT 'evidence',
  ROUND(MIN(evidence_score)), ROUND(PERCENTILE(evidence_score, 0.25)),
  ROUND(PERCENTILE(evidence_score, 0.50)), ROUND(PERCENTILE(evidence_score, 0.75)),
  ROUND(MAX(evidence_score)), ROUND(AVG(evidence_score), 1),
  SUM(CASE WHEN evidence_score = 100 THEN 1 END),
  SUM(CASE WHEN evidence_score = 0   THEN 1 END)
FROM genealogy.gold_person_scores
UNION ALL
SELECT 'story_potential',
  ROUND(MIN(story_potential_score)), ROUND(PERCENTILE(story_potential_score, 0.25)),
  ROUND(PERCENTILE(story_potential_score, 0.50)), ROUND(PERCENTILE(story_potential_score, 0.75)),
  ROUND(MAX(story_potential_score)), ROUND(AVG(story_potential_score), 1),
  SUM(CASE WHEN story_potential_score = 100 THEN 1 END),
  SUM(CASE WHEN story_potential_score = 0   THEN 1 END)
FROM genealogy.gold_person_scores
ORDER BY dimension;


In [0]:

%sql
SELECT * FROM genealogy.gold_branch_scores;


In [0]:

%sql
-- Spot-check: confirm direct ancestors rank above distant relatives
-- with comparable raw scores. Sample top 20 by weighted_overall_score
-- alongside their absolute score and multipliers for sanity checking.
SELECT
  given_name,
  surname,
  birth_year,
  branch,
  proximity_label,
  ancestral_proximity,
  proximity_multiplier,
  depth_multiplier,
  overall_score,
  weighted_overall_score,
  weighted_story_potential_score
FROM genealogy.gold_person_scores
ORDER BY weighted_overall_score ASC  -- ASC = lowest weighted score = most research needed
LIMIT 20;


In [0]:

%sql
-- Cross-join on thresholds to find the right story_ready cutoff.
-- Aim for ~50 story-ready individuals with good direct-ancestor coverage.
WITH thresholds AS (
  SELECT explode(sequence(60, 100, 1)) AS threshold
),
scores AS (
  SELECT person_gedcom_id, story_potential_score, weighted_story_potential_score, story_written, proximity_label, branch
  FROM genealogy.gold_person_scores
  WHERE NOT COALESCE(story_written, FALSE)
)
SELECT
  t.threshold,
  COUNT(*)                                                        AS total_story_ready,
  SUM(CASE WHEN s.proximity_label = 'Direct Ancestor' THEN 1 ELSE 0 END) AS direct_ancestors,
  COUNT(DISTINCT s.branch)                                        AS branches_covered
FROM thresholds t
--JOIN scores s ON s.story_potential_score >= t.threshold
JOIN scores s ON s.weighted_story_potential_score >= t.threshold
GROUP BY t.threshold
ORDER BY t.threshold;
